# STEP 1 — Load SPM List

In [1]:
import pandas as pd

spm_df = pd.read_excel("/content/Types_of_SPMs.xlsx")
spm_list = (
    spm_df.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

print(f"Loaded {len(spm_list)} SPMs")

Loaded 57 SPMs


In [6]:
spm_lookup = {}

for spm in spm_list:
    spm_lookup[spm.lower()] = spm

In [7]:
def normalize_spm(name):

    if not name:
        return None

    name_lower = name.lower()

    for spm in spm_lookup:
        if spm in name_lower:
            return spm_lookup[spm]

    return name

# STEP 2 — PubMed Retrieval

In [8]:
import requests
import time

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
EMAIL = "venkat.boddu@uni-rostock.de"
  # optional but recommended

def search_pubmed(query, batch_size=200):
    all_ids = []
    retstart = 0

    while True:
        params = {
            "db": "pubmed",
            "term": query,
            "retmax": batch_size,
            "retstart": retstart,
            "retmode": "json",
            "email": EMAIL
        }

        r = requests.get(BASE_URL + "esearch.fcgi", params=params)

        if r.status_code != 200:
            print("HTTP error for:", query)
            print(r.text[:300])
            break

        if "application/json" not in r.headers.get("Content-Type", ""):
            print("Non-JSON response for:", query)
            print(r.text[:300])
            break

        try:
            data = r.json()["esearchresult"]
        except Exception:
            print("JSON parsing failed for:", query)
            print(r.text[:300])
            break

        ids = data["idlist"]
        if not ids:
            break

        all_ids.extend(ids)
        retstart += batch_size
        time.sleep(0.34)

    return all_ids

# STEP 3 — Retrieve Articles for All SPMs

In [9]:
def get_count(query):
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "rettype": "count",
        "email": EMAIL
    }

    r = requests.get(BASE_URL + "esearch.fcgi", params=params)
    r.raise_for_status()
    return int(r.json()["esearchresult"]["count"])

In [10]:
def search_pubmed_latest(spm, max_results=1000):
    query = f'"{spm}"[Title/Abstract]'

    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "sort": "pub date",     # newest first
        "datetype": "pdat",     # publication date
        "email": EMAIL
    }

    r = requests.get(BASE_URL + "esearch.fcgi", params=params)
    r.raise_for_status()

    data = r.json()
    return data["esearchresult"]["idlist"]



In [11]:
all_pmids = set()
max_limit_for_pubmet_records=200

for spm in spm_list:
    print(f"Searching latest {max_limit_for_pubmet_records} for: {spm}")

    try:
        pmids = search_pubmed_latest(spm, max_limit_for_pubmet_records)
        print(f"   Retrieved: {len(pmids)}")

        all_pmids.update(pmids)

        time.sleep(0.34)  # safe without NCBI key

    except Exception as e:
        print("Failed:", spm, e)
        continue

print("Total unique PMIDs collected:", len(all_pmids))

Searching latest 200 for: MCTR1
   Retrieved: 24
Searching latest 200 for: MCTR2
   Retrieved: 3
Searching latest 200 for: MCTR3
   Retrieved: 12
Searching latest 200 for: PCTR1
   Retrieved: 21
Searching latest 200 for: PCTR2
   Retrieved: 6
Searching latest 200 for: PCTR3
   Retrieved: 1
Searching latest 200 for: RCTR1
   Retrieved: 8
Searching latest 200 for: RCTR2
   Retrieved: 1
Searching latest 200 for: RCTR3
   Retrieved: 2
Searching latest 200 for: LXA4
   Retrieved: 200
Searching latest 200 for: LXA5
   Retrieved: 13
Searching latest 200 for: LXB4
   Retrieved: 113
Searching latest 200 for: LXB5
   Retrieved: 3
Searching latest 200 for: 15-epi-LXA4
   Retrieved: 60
Searching latest 200 for: 15-epi-LXB4
   Retrieved: 4
Searching latest 200 for: 7-epi-MaR1
   Retrieved: 0
Searching latest 200 for: eMaR
   Retrieved: 87
Searching latest 200 for: MaR-L1
   Retrieved: 0
Searching latest 200 for: MaR-L2
   Retrieved: 0
Searching latest 200 for: MaR1
   Retrieved: 200
Searching lates

## fetch abstracts

In [12]:
def fetch_abstract(pmid):
    params = {
        "db": "pubmed",
        "id": pmid,
        "rettype": "abstract",
        "retmode": "text",
        "email": EMAIL
    }

    r = requests.get(BASE_URL + "efetch.fcgi", params=params)
    r.raise_for_status()
    return r.text

In [13]:
articles = {}

for pmid in list(all_pmids):
    try:
        articles[pmid] = fetch_abstract(pmid)
        time.sleep(0.11)
    except:
        continue

In [14]:
print(len(articles))
print(len(all_pmids))

1798
1799


# STEP 4 — Chunking for LLM Safety

In [15]:
def chunk_text(text, max_chars=24000):
    chunks = []
    for i in range(0, len(text), max_chars):
        chunks.append(text[i:i+max_chars])
    return chunks

# STEP 5 — LLM Extraction (JSON Structured Output)

In [16]:
# pip install openai

In [17]:
spm_list = sorted(set(spm_df["Name"].dropna().astype(str)))

SPM_TEXT = "\n".join(spm_list)

In [18]:
from openai import OpenAI
LLAMA_API_KEY = "607b888aa4705f2789951f62edbe35b3"


client = OpenAI(
    base_url="https://chat-ai.academiccloud.de/v1",
    api_key=LLAMA_API_KEY
)


In [25]:
# API_URL = "https://chat-ai.academiccloud.de/v1/chat/completions"
# API_KEY_LLAMA = "YOUR_ACADEMIC_CLOUD_KEY"
API_KEY_LLAMA = "607b888aa4705f2789951f62edbe35b3"
API_URL = "https://chat-ai.academiccloud.de/v1"


SYSTEM_PROMPT = """
You are a biomedical information extraction system.

Extract interactions between Specialized Pro-Resolving Mediators (SPMs) and proteins
from the following PubMed abstract.

Valid SPM names (use ONLY these):
{SPM_TEXT}

Return ONLY valid JSON.

Output format:

[
  {
    "SPM": "name",
    "Relation": "upregulates | downregulates | binds | modulates",
    "Target_Protein": "protein name",
    "Evidence": "exact sentence from the abstract",
    "PMID": "pmid"
  }
]

Rules:
- Relation MUST be one of: upregulates, downregulates, binds, modulates. If the paper used similar phrase, map it to closest allowed relation
- Evidence must be the eact sentence from the abstract
- do not invent proteins or SPMs, use the SPM name from list of valid SPM

- If no interaction exists, return []

Return ONLY JSON. No explanations.
"""



def call_llama(text_chunk):
    headers = {
        "Authorization": f"Bearer {API_KEY_LLAMA}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.1-8b-instruct",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text_chunk}
        ],
        "temperature": 0,
        "max_tokens": 2000
    }

    r = requests.post(API_URL, headers=headers, json=payload)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

In [26]:
import time

def call_llama(text_chunk, retries=5):

    wait = 2

    for attempt in range(retries):

        try:

            response = client.chat.completions.create(
                model="meta-llama-3.1-8b-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text_chunk}
                ],
                temperature=0,
                max_tokens=1200
            )

            return response.choices[0].message.content

        except Exception as e:

            if "429" in str(e):
                print(f"Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
                wait *= 2
                continue

            elif "404" in str(e):
                print("Model endpoint temporarily unavailable. Retrying...")
                time.sleep(wait)
                wait *= 2
                continue

            else:
                raise e

    return None

# STEP 6 — Extract Interactions

In [29]:
import json
import re

def recover_json_rows(text):
    """
    Extract individual JSON objects even if the full JSON is broken.
    Returns list of valid dict rows.
    """

    rows = []

    # find every { ... } block
    objects = re.findall(r"\{.*?\}", text, re.DOTALL)

    for obj in objects:
        try:
            cleaned = obj.replace("\n", " ")
            rows.append(json.loads(cleaned))
        except:
            continue

    return rows

In [ ]:
# STEP 6 — Extract Interactions

import json
import re
import pandas as pd

ALLOWED_RELATIONS = {"upregulates","downregulates","binds","modulates"}

def recover_json_rows(text):

    rows = []

    # remove markdown formatting
    text = text.replace("```json", "").replace("```", "")

    # remove newlines that break JSON
    text = text.replace("\n"," ")

    objects = re.findall(r"\{.*?\}", text)

    for obj in objects:

        try:
            data = json.loads(obj)

            if all(k in data for k in ["SPM","Relation","Target_Protein","Evidence","PMID"]):

                if data["Relation"] in ALLOWED_RELATIONS:

                    data["SPM"] = normalize_spm(data.get("SPM"))

                    rows.append(data)

        except:
            continue

    return rows


all_interactions = []

batch_size = 10
items = list(articles.items())

for i in range(0, len(items), batch_size):

    batch = items[i:i+batch_size]

    print(f"Processing batch {i} to {i+batch_size}")

    texts = []

    for pmid, article in batch:

        chunks = chunk_text(article)

        for chunk in chunks:
            texts.append(f"PMID:{pmid}\n{chunk}")

    combined_text = "\n\n---ARTICLE---\n\n".join(texts)

    try:

        raw_output = call_llama(combined_text)

        rows = recover_json_rows(raw_output)

        if len(rows) == 0:
            print("No valid rows recovered")

        else:
            all_interactions.extend(rows)
            print(f"Recovered {len(rows)} interactions")

    except Exception as e:

        print("Batch failed:", e)
        continue


print("Total extracted interactions:", len(all_interactions))

Processing batch 0 to 10
Recovered 4 interactions
Processing batch 10 to 20
Recovered 1 interactions
Processing batch 20 to 30
Recovered 14 interactions
Processing batch 30 to 40
Recovered 10 interactions
Processing batch 40 to 50
Recovered 13 interactions
Processing batch 50 to 60
Recovered 1 interactions
Processing batch 60 to 70
Recovered 13 interactions
Processing batch 70 to 80
Recovered 13 interactions
Processing batch 80 to 90
Recovered 12 interactions
Processing batch 90 to 100
Recovered 15 interactions
Processing batch 100 to 110
Recovered 9 interactions
Processing batch 110 to 120
Recovered 8 interactions
Processing batch 120 to 130
Recovered 1 interactions
Processing batch 130 to 140
Recovered 9 interactions
Processing batch 140 to 150
No valid rows recovered
Processing batch 150 to 160
Recovered 7 interactions
Processing batch 160 to 170
Recovered 15 interactions
Processing batch 170 to 180
Recovered 4 interactions
Processing batch 180 to 190
Recovered 2 interactions
Proces

In [31]:

for i in range(1160, len(items), batch_size):

    batch = items[i:i+batch_size]

    print(f"Processing batch {i} to {i+batch_size}")

    texts = []

    for pmid, article in batch:

        chunks = chunk_text(article)

        for chunk in chunks:
            texts.append(f"PMID:{pmid}\n{chunk}")

    combined_text = "\n\n---ARTICLE---\n\n".join(texts)

    try:

        raw_output = call_llama(combined_text)

        rows = recover_json_rows(raw_output)

        if len(rows) == 0:
            print("No valid rows recovered")

        else:
            all_interactions.extend(rows)
            print(f"Recovered {len(rows)} interactions")

    except Exception as e:

        print("Batch failed:", e)
        continue


print("Total extracted interactions:", len(all_interactions))



Processing batch 1160 to 1170
No valid rows recovered
Processing batch 1170 to 1180
Recovered 4 interactions
Processing batch 1180 to 1190
Recovered 11 interactions
Processing batch 1190 to 1200
Recovered 6 interactions
Processing batch 1200 to 1210
Recovered 2 interactions
Processing batch 1210 to 1220
Recovered 4 interactions
Processing batch 1220 to 1230
Recovered 11 interactions
Processing batch 1230 to 1240
Recovered 11 interactions
Processing batch 1240 to 1250
Recovered 2 interactions
Processing batch 1250 to 1260
No valid rows recovered
Processing batch 1260 to 1270
Recovered 7 interactions
Processing batch 1270 to 1280
Recovered 11 interactions
Processing batch 1280 to 1290
Recovered 5 interactions
Processing batch 1290 to 1300
Recovered 4 interactions
Processing batch 1300 to 1310
Recovered 8 interactions
Processing batch 1310 to 1320
Recovered 4 interactions
Processing batch 1320 to 1330
Recovered 1 interactions
Processing batch 1330 to 1340
No valid rows recovered
Processin

# STEP 7 — Save Clean CSV

In [32]:
import pandas as pd

df = pd.DataFrame(all_interactions)
df.to_csv("/content/llm_parsed_output_17-mar.csv", index=False)

print("Saved:", len(df), "rows")

Saved: 1364 rows


## Step 8 - Filter output and order the CSV results

In [33]:
import pandas as pd

df = pd.read_csv("/content/llm_parsed_output_17-mar.csv")

In [34]:
spm_order = {spm: i for i, spm in enumerate(spm_list)}

In [35]:
## order SPM according to Types of SPM list
df["spm_order"] = df["SPM"].map(spm_order)

df = df.sort_values(["spm_order"])

df = df.drop(columns=["spm_order"])

In [36]:
# Remove rows where Target_Protein is null
df = df[df["Target_Protein"].notna()]
df = df[df["Target_Protein"].str.strip() != ""]

In [37]:
df.to_csv("/content/result_17-mar.csv", index=False)